# Is That a Peak or a Cosmic Ray? Cleaning Raman Properly
### 4.5 — Raman: Cosmic Ray Removal and Fluorescence Baselines

A raw Raman spectrum is a small signal wrapped in two big problems. The sharp,
information-rich Raman bands you actually want are usually riding on a **broad
fluorescence background** that can be larger than the bands themselves — and
scattered through the spectrum are **cosmic-ray spikes**: razor-thin, sky-high
counts that have nothing to do with the sample at all. Both can fool you. A tall
cosmic ray *looks* like a strong peak; a fluorescence hump *looks* like a broad
band. Clean them wrong and you will either invent chemistry that isn't there or
erase chemistry that is.

This notebook is **not** "call `despike()` and `baseline()` and move on." It
builds the reasoning:

- **why** Raman spectra so often carry a fluorescence background, and **why**
  cosmic rays appear as sharp, non-chemical spikes;
- how each artifact **masquerades as signal**;
- why **despiking and baseline correction are different problems** that need
  different tools;
- how to **identify** cosmic rays from their *narrowness* and *intensity*, and
  remove them **without flattening real Raman bands**;
- how an **AsLS-style** baseline removes the broad fluorescence;
- why the **order** of these steps changes the answer;
- and what **over-cleaning** does — measured, on purpose.

Because every spectrum comes from `raman.simulate()`, we know the **true** band
profile, the **true** fluorescence curve, and the **exact indices** of every
injected spike. That answer key — which real data can never give you — lets us
*grade* each step instead of admiring it.

> **The one idea to hold onto.** Cosmic rays and fluorescence sit at **opposite
> ends of the width scale**: a cosmic ray is *impossibly narrow* (one or two
> pixels), fluorescence is *impossibly broad* (the whole axis). A real Raman
> band lives in between. That width gap is the entire game — it's how you tell
> each artifact from the chemistry, and it's why the tool that removes one is
> the *wrong* tool for the other. Despiking is a **local, high-frequency** fix;
> baseline correction is a **global, low-frequency** fix. Never use one to do the
> other's job.

## 1. Setup

The standard scientific-Python stack plus our project's Raman simulator. We
build AsLS from scratch with SciPy's sparse solver (same engine as lesson 3.3),
so the baseline machinery stays visible rather than hidden in a black box.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve

# Our Raman simulator. Simulated data carries its own ground truth: the true
# Lorentzian band profile, the true fluorescence curve, and the exact indices of
# every injected cosmic ray -- so we can grade every cleaning step honestly.
from simulated_data import raman

# Figures are regenerable scratch; the exports/ folder is git-ignored.
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

SEED = 11                      # fix every random draw so your output matches ours
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

## 2. Why Raman carries fluorescence *and* cosmic rays

Raman scattering is a **weak** effect — only about one in ten million photons
scatters inelastically and carries the vibrational fingerprint you want. That
weakness is the root of both problems.

**Fluorescence.** The same laser that drives Raman scattering also *excites
electrons*. If the sample (or a trace impurity, or the substrate) is even mildly
fluorescent, it absorbs laser photons and re-emits them over a **broad, smooth
band** of wavelengths. Because fluorescence is a far more probable process than
Raman scattering, that broad emission is frequently **much larger** than the
sharp Raman lines sitting on top of it. It is not noise and not an instrument
fault — it is real light from the sample, just not the light you came for. Its
defining feature is that it is **slow and broad**: it varies smoothly across the
whole spectrum.

**Cosmic rays.** Raman detectors are CCDs run at long exposures to collect those
rare Raman photons. During an exposure a high-energy particle — a cosmic ray, or
a stray gamma from the environment — can strike the chip and dump a burst of
charge into **one or two pixels**. The result is a **single-pixel spike**, often
taller than any real peak, appearing at a *random* position that **changes every
exposure**. Its defining feature is the opposite of fluorescence: it is
**impossibly narrow**.

So the two artifacts are physical opposites — broad-and-smooth versus
sharp-and-narrow — and that is exactly what lets us separate them from the
finite-width Raman bands in between.

## 3. A realistic Raman spectrum with known ground truth

We simulate one spectrum over the fingerprint + C–H region (200–3200 cm⁻¹) at
high resolution, with all three real-world ingredients switched on:

- **sharp Lorentzian bands** — the natural Raman line shape;
- a **broad fluorescence background** (`fluorescence=3.0`) that dwarfs the bands;
- **cosmic-ray spikes** (`cosmic_rays=4`) at random pixels;
- **detector noise**.

The high point count (4000) matters scientifically: it keeps the real bands
*well sampled* (each FWHM spans ~13+ points), so a real peak rises gradually
from point to point. A cosmic ray, by contrast, jumps in a **single** point.
That sampling gap is what makes the spikes detectable later without touching the
bands — under-sample your spectrum and you lose the ability to tell them apart.

`raman.simulate()` records the answer key in `meta`:
`meta.attrs["clean_peaks"]` (true bands, no background/noise/spikes),
`meta.attrs["fluorescence_curve"]` (true background), and
`meta["cosmic_ray_indices"]` (exact spike positions).

In [ ]:
ds = raman.simulate(
    fluorescence=3.0,     # broad background, ~5x taller than the bands
    cosmic_rays=4,        # four single-pixel spikes at random positions
    noise=0.02,           # modest detector noise
    n_samples=1,
    seed=SEED,
    n_points=4000,        # high resolution -> real bands are well sampled
)

x = ds.x                                      # Raman shift axis (cm-1)
raw = ds.X[0]                                 # the raw measured spectrum

# --- The ground truth the simulator handed us (real data can't do this) ---
true_peaks = ds.meta.attrs["clean_peaks"]            # bands only
true_fluor = ds.meta.attrs["fluorescence_curve"]     # background only
true_spikes = np.asarray(ds.meta["cosmic_ray_indices"].iloc[0])  # spike indices

spacing = (x[-1] - x[0]) / (len(x) - 1)
print("axis            : %.0f-%.0f %s, %d points (%.2f cm-1/point)"
      % (x[0], x[-1], ds.x_unit, len(x), spacing))
print("band height max : %.2f   fluorescence max : %.2f   raw max : %.2f"
      % (true_peaks.max(), true_fluor.max(), raw.max()))
print("true spike index:", true_spikes, "-> shifts (cm-1):",
      np.round(x[true_spikes]).astype(int))

## 4. The raw spectrum: what the instrument actually hands you

Plot the raw spectrum and the two artifacts jump out immediately — which is the
problem. The eye is drawn to the **tallest** features, and here the tallest
features are *artifacts*, not chemistry.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x, raw, lw=0.8, color="#1f77b4", label="raw Raman spectrum")
ax.plot(x, true_fluor, lw=2, color="#d62728", ls="--",
        label="true fluorescence background")

# Mark where the cosmic rays really are.
ax.plot(x[true_spikes], raw[true_spikes], "v", color="black", ms=9,
        label="cosmic rays (truth)")

ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title("Raw Raman: sharp spikes + broad fluorescence bury the real bands")
ax.legend(loc="upper right", framealpha=0.95)
fig.tight_layout()
fig.savefig(EXPORTS / "01_raw_spectrum.png", dpi=150)
plt.show()

Two things to notice. First, the **broad red swell** — the fluorescence —
carries the whole spectrum far off the zero line and varies smoothly across the
axis. Second, the **black triangles** mark four spikes, several of them taller
than anything chemical in the spectrum. The actual Raman bands? Small ripples on
the shoulder of the swell. This is the normal Raman starting point, not a
worst case.

## 5. How both artifacts masquerade as signal

Zoom in and the disguise becomes obvious. A cosmic ray, magnified, looks like a
**strong, narrow peak**. A stretch of fluorescence looks like a **broad band**.
Neither is chemistry — but a peak-picker that only knows "tall = important" will
report both.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

# --- Left: a cosmic ray next to a real band, same y-scale ---
spike = true_spikes[np.argmax(raw[true_spikes])]      # the tallest spike
band = int(np.argmin(np.abs(x - 1000.0)))             # the strong 1000 cm-1 band
ax = axes[0]
half = 120
sl = slice(max(0, spike - half), spike + half)
ax.plot(x[sl], raw[sl], lw=1.0, color="#1f77b4")
ax.plot(x[spike], raw[spike], "v", color="black", ms=10)
ax.annotate("cosmic ray\n(1 pixel, no width)", (x[spike], raw[spike]),
            textcoords="offset points", xytext=(10, -28))
ax.set_title("A cosmic ray masquerades as a tall peak")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)

# --- Right: the broad fluorescence shaped like a band ---
ax = axes[1]
ax.plot(x, raw, lw=0.7, color="#1f77b4", label="raw")
ax.fill_between(x, 0, true_fluor, color="#d62728", alpha=0.15,
                label="fluorescence (broad 'band')")
ax.set_title("Fluorescence masquerades as a broad band")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)
ax.legend(loc="upper right")

fig.tight_layout()
fig.savefig(EXPORTS / "02_masquerade.png", dpi=150)
plt.show()

The contrast in the **left panel** is the key to everything that follows: the
real band has a *finite width* — it rises and falls over many points — while the
cosmic ray is a vertical line, present at essentially one pixel. Width, not
height, is the tell.

## 6. Why despiking and baseline correction are different problems

It is tempting to reach for one tool to "clean the spectrum." Don't. The two
artifacts live at opposite ends of the frequency scale, so they need opposite
tools:

| | Cosmic ray | Fluorescence |
|---|---|---|
| **Width** | 1–2 pixels (impossibly narrow) | whole axis (impossibly broad) |
| **In frequency terms** | very **high** frequency | very **low** frequency |
| **Position** | random, changes every scan | fixed, smooth, reproducible |
| **Right tool** | **despiking** (local outlier removal) | **baseline correction** (smooth fit) |
| **Wrong tool** | a smooth baseline *ignores* it | a despiker *can't see* a slow swell |

A smooth baseline fit (AsLS) is *designed* to follow slow trends — it glides
right under a one-pixel spike as if it weren't there, leaving the spike in your
"corrected" data. A despiker looks for **local** outliers — it has no concept of
a trend that takes 3000 cm⁻¹ to develop. Each tool is blind to the other's
target. That's why you need both, and why mixing them up fails.

## 7. Detecting cosmic rays by narrowness *and* intensity

How do we turn "impossibly narrow and tall" into a number? Take the
**point-to-point difference** of the spectrum, `∇y[i] = y[i] − y[i−1]`. A real
band, being well sampled, changes only a little between adjacent points — small
∇y. A cosmic ray jumps up in a single step and back down in the next — a huge
positive ∇y immediately followed by a huge negative one. The difference series
*amplifies narrowness*.

To flag "huge" robustly (without letting the spikes themselves set the scale) we
use the **modified z-score**, built on the **median** and the **median absolute
deviation (MAD)** instead of the mean and standard deviation — both of which a
few giant spikes would wreck:

$$z_i = \frac{0.6745\,(\nabla y_i - \mathrm{median}(\nabla y))}{\mathrm{MAD}(\nabla y)}$$

A point with $|z_i|$ above a threshold (≈ 7) is a spike. This is the classic
Whitaker–Hayes despiking criterion, and it encodes **both** of our clues at
once: the difference captures **narrowness**, the z-score captures **intensity**.

In [ ]:
def modified_z(values):
    '''Robust z-score using the median and MAD (resistant to the spikes themselves).'''
    med = np.median(values)
    mad = np.median(np.abs(values - med))
    mad = mad if mad > 0 else 1e-12          # guard the (degenerate) flat case
    return 0.6745 * (values - med) / mad


def spike_zscores(y):
    '''|modified z-score| of the point-to-point difference -- one value per point.

    ``prepend=y[0]`` keeps the output the same length as ``y`` (the first
    difference is defined as 0), so an index into ``z`` is an index into ``y``.
    '''
    diff = np.diff(y, prepend=y[0])
    return np.abs(modified_z(diff))


THRESHOLD = 7.0
z = spike_zscores(raw)
flagged = z > THRESHOLD                       # boolean mask, one per point
detected = np.where(flagged)[0]

print("points flagged   :", detected, "-> shifts:", np.round(x[detected]).astype(int))
print("true spike points:", true_spikes)

Each cosmic ray trips the detector at **two** adjacent points — the step *up* and
the step back *down* — which is why you see pairs. That's harmless: we'll repair
both. Now plot the z-scores so the separation between artifact and chemistry is
visible.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(x, raw, lw=0.8, color="#1f77b4")
axes[0].plot(x[detected], raw[detected], "o", mfc="none", mec="red", ms=9,
             label="flagged as spike")
axes[0].set_ylabel(ds.y_label); axes[0].legend(loc="upper right")
axes[0].set_title("Spike detection: the z-score of the difference separates artifact from chemistry")

axes[1].plot(x, z, lw=0.7, color="#555")
axes[1].axhline(THRESHOLD, color="red", ls="--", label=f"threshold = {THRESHOLD}")
axes[1].set_ylabel("|modified z|"); axes[1].set_xlabel(f"{ds.x_label} ({ds.x_unit})")
axes[1].legend(loc="upper right")
fig.tight_layout()
fig.savefig(EXPORTS / "03_spike_detection.png", dpi=150)
plt.show()

In [ ]:
# --- Grade the detector against the known spike positions -------------------
# A true spike counts as "caught" if any flagged point lands within 1 pixel of it.
caught = sum(any(abs(detected - t) <= 1) for t in true_spikes)
# A flagged point is a false alarm if it's not within 1 pixel of any true spike.
false_alarms = sum(all(abs(d - true_spikes) > 1) for d in detected)
recall = caught / len(true_spikes)

# How far below threshold did the real chemistry stay? (the safety margin)
mask = np.ones(len(x), bool)
for t in true_spikes:
    mask[max(0, t - 2): t + 3] = False        # exclude the spikes themselves
peak_z_max = z[mask].max()

print("recall (true spikes caught) : %d/%d = %.0f%%" % (caught, len(true_spikes), 100 * recall))
print("false alarms                : %d" % false_alarms)
print("largest z on real chemistry : %.2f  (threshold %.1f -> margin %.2f)"
      % (peak_z_max, THRESHOLD, THRESHOLD - peak_z_max))

All four spikes caught, zero false alarms, and the loudest *real* feature only
reached a z of about 4.6 — comfortably under the threshold of 7. That margin is
the payoff of well-sampled bands: there is clear daylight between "narrow
artifact" and "real chemistry."

## 8. Despiking without destroying real bands

Detection is half the job. The repair must remove the flagged points **and
nothing else** — replace each spike by interpolating across it from its healthy
neighbors, leaving every unflagged point (i.e. all the real chemistry) exactly
as measured. This is the crucial discipline: a despiker should be a *scalpel*,
touching only the handful of corrupted pixels, never a blanket smoother that
rounds off every sharp band in the spectrum.

In [ ]:
def despike(y, threshold=THRESHOLD):
    '''Remove cosmic rays by linear interpolation across flagged points only.

    Returns the cleaned spectrum and the boolean mask of points it repaired.
    Every *unflagged* point is left bit-for-bit unchanged -- the real bands are
    never touched.
    '''
    flagged = spike_zscores(y) > threshold
    out = y.copy()
    if flagged.any():
        good = np.where(~flagged)[0]                 # the trustworthy points
        bad = np.where(flagged)[0]
        out[bad] = np.interp(bad, good, y[good])     # bridge each gap
    return out, flagged


despiked, mask = despike(raw)

# Proof that only the spikes moved: every non-flagged point is identical.
untouched = np.array_equal(despiked[~mask], raw[~mask])
print("non-spike points left exactly unchanged :", untouched)
print("points repaired                         :", int(mask.sum()), "of", len(x))

In [ ]:
# Zoom on the strong 1000 cm-1 band to show it survives intact, and on a spike
# to show it's gone.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

ax = axes[0]
sl = slice(band - 80, band + 80)
ax.plot(x[sl], raw[sl], lw=1.4, color="#1f77b4", label="raw")
ax.plot(x[sl], despiked[sl], lw=1.4, color="#2ca02c", ls="--", label="despiked")
ax.set_title("Real band at 1000 cm$^{-1}$ is preserved exactly")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label); ax.legend()

ax = axes[1]
spike = true_spikes[np.argmax(raw[true_spikes])]
sl = slice(spike - 80, spike + 80)
ax.plot(x[sl], raw[sl], lw=1.2, color="#1f77b4", label="raw")
ax.plot(x[sl], despiked[sl], lw=1.6, color="#2ca02c", label="despiked")
ax.set_title("Cosmic ray removed, baseline underneath untouched")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label); ax.legend()

fig.tight_layout()
fig.savefig(EXPORTS / "04_despiked.png", dpi=150)
plt.show()

The green despiked trace lies exactly on top of the raw band on the left (we
verified it is bit-for-bit identical away from spikes) while the spike on the
right is gone, with the broad fluorescence beneath it left completely alone — as
it should be, because removing the fluorescence is a *different job*.

## 9. Estimating the fluorescence baseline with AsLS

With the spikes gone we can tackle the broad swell. **Asymmetric Least Squares
(AsLS)** — the workhorse from lesson 3.3 — fits a smooth curve that is pulled
toward the spectrum but penalized for bending (a smoothness term, `λ`) and
allowed to sit **under** the peaks rather than through them (an asymmetry term,
`p`, that gives points *above* the current fit very little weight). That is
exactly the right model for fluorescence: a smooth thing that lives *below* the
sharp Raman bands.

We must run AsLS on the **despiked** spectrum. A leftover cosmic ray is a huge
point the fit would either chase or be distorted by — another reason despiking
comes first.

In [ ]:
def asls_baseline(y, lam=1e6, p=0.01, n_iter=10):
    '''Asymmetric Least Squares baseline (Eilers & Boelens), built from scratch.

    lam : smoothness penalty -- larger gives a stiffer, broader baseline.
    p   : asymmetry -- small p (0.001-0.05) tells the fit to ignore the peaks
          and hug the valleys, which is what we want for fluorescence.
    '''
    L = len(y)
    # Second-difference operator; penalizing its square is what enforces smoothness.
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    penalty = lam * (D @ D.T)
    w = np.ones(L)
    z = np.zeros(L)
    for _ in range(n_iter):
        W = sparse.spdiags(w, 0, L, L)
        z = spsolve(W + penalty, w * y)
        w = p * (y > z) + (1.0 - p) * (y < z)     # down-weight points above the fit
    return z


LAM, P = 1e6, 0.01
baseline = asls_baseline(despiked, lam=LAM, p=P)

# Grade the estimate against the TRUE fluorescence curve.
base_rmse = np.sqrt(np.mean((baseline - true_fluor) ** 2))
print("AsLS baseline vs TRUE fluorescence : RMSE = %.4f  (background height %.2f)"
      % (base_rmse, true_fluor.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x, despiked, lw=0.8, color="#1f77b4", label="despiked spectrum")
ax.plot(x, baseline, lw=2.2, color="#2ca02c", label=f"AsLS baseline (λ={LAM:.0e}, p={P})")
ax.plot(x, true_fluor, lw=2.0, color="#d62728", ls=":", label="true fluorescence")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)
ax.set_title("AsLS recovers the broad fluorescence, gliding under the sharp bands")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(EXPORTS / "05_baseline_fit.png", dpi=150)
plt.show()

The green AsLS estimate tracks the red truth across the whole axis and passes
*beneath* the Raman bands instead of carving into them — RMSE of a few percent of
the background height. That's a faithful fluorescence estimate, ready to
subtract.

## 10. Subtraction → corrected spectrum, graded against truth

Subtract the baseline and compare what's left to the **true band profile** the
simulator used. This is the moment of truth: did the full clean recover the
chemistry?

In [ ]:
corrected = despiked - baseline

corr_rmse = np.sqrt(np.mean((corrected - true_peaks) ** 2))
print("corrected spectrum vs TRUE bands : RMSE = %.4f  (band height %.2f)"
      % (corr_rmse, true_peaks.max()))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x, corrected, lw=1.0, color="#1f77b4", label="cleaned (despiked + baseline-subtracted)")
ax.plot(x, true_peaks, lw=1.8, color="#d62728", ls="--", label="true bands (ground truth)")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)
ax.set_title("Cleaned spectrum vs known truth: the chemistry is back")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(EXPORTS / "06_corrected_vs_truth.png", dpi=150)
plt.show()

The cleaned blue trace sits on the dashed red truth: five Lorentzian bands on a
flat zero baseline, no spikes, residual error only at the level of the detector
noise. We started with artifacts taller than the signal and ended within a few
percent of the right answer — and we can *say* that quantitatively only because
the data carried its own ground truth.

## 11. Why the order matters

"Despike, then baseline" is not arbitrary. The rule is: **despike first, before
any step that smooths or spreads the data.** Here's the trap. A natural instinct
is to smooth a noisy Raman spectrum early. But smoothing a one-pixel cosmic ray
**spreads it into a several-pixel bump** — and a bump is no longer narrow, so the
despiker can't recognize it. The spike survives, disguised as a fake Raman band
*taller than the real ones*.

In [ ]:
def boxcar(y, w=5):
    '''A simple moving-average smoother (the kind beginners reach for first).'''
    return np.convolve(y, np.ones(w) / w, mode="same")

# Right order: despike FIRST, then smooth.
right = boxcar(despike(raw)[0])

# Wrong order: smooth FIRST, then despike.
wrong, _ = despike(boxcar(raw))

# Both traces still carry the (unremoved) fluorescence + smoothed bands, so we
# isolate the SURVIVING SPIKE as the difference between the two orders at the
# known spike sites: where the wrong order left a bump, the right order is flat.
surviving_spike = max(abs(wrong[t] - right[t]) for t in true_spikes)
print("surviving cosmic-ray bump (wrong order minus right order):")
print("  height = %.3f  <- a fake 'band', and real Raman bands peak at only %.2f"
      % (surviving_spike, true_peaks.max()))

In [ ]:
spike = true_spikes[np.argmax(raw[true_spikes])]
sl = slice(spike - 120, spike + 120)
fig, ax = plt.subplots(figsize=(11, 4.3))
ax.plot(x[sl], wrong[sl], lw=1.6, color="#d62728",
        label="smooth → despike (WRONG): fake band survives")
ax.plot(x[sl], right[sl], lw=1.6, color="#2ca02c",
        label="despike → smooth (RIGHT): spike gone")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)
ax.set_title("Order matters: smoothing first turns a removable spike into a fake peak")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(EXPORTS / "07_order_matters.png", dpi=150)
plt.show()

Smoothing first leaves a broad fake band where the cosmic ray was — taller than
any real peak — because the smoother destroyed the very narrowness the despiker
relies on. The same logic puts **despiking before baseline correction**: never
let an artifact bias a fit, and never let a smoothing step blur an artifact you
haven't removed yet. **Despike first. Always.**

## 12. Failure cases: over-despiking and over-baseline-correction

Both tools have an aggressiveness knob, and turning it up too far does *more*
damage than leaving the artifact in — because now you're deleting real
chemistry. We make both failures happen on purpose and measure them.

**Over-despiking** — threshold too low and a replacement window too wide — starts
flagging the flanks of real bands as "outliers" and interpolating across them,
eroding genuine peak area. **Over-baseline-correction** — `λ` too small — lets
the baseline stop being stiff and **bend up into the peaks**, so subtraction
carves the bands away: the band heights and areas shrink toward zero.

In [ ]:
# --- Over-despiking: low threshold + wide replacement window ---------------
def despike_aggressive(y, threshold=2.0, win=5):
    '''A deliberately over-eager despiker: low threshold, wide gouged window.'''
    flagged = spike_zscores(y) > threshold
    idx = np.where(flagged)[0]
    for i in idx:                                  # widen each hit into a window
        flagged[max(0, i - win): i + win + 1] = True
    out = y.copy()
    good = np.where(~flagged)[0]
    out[flagged] = np.interp(np.where(flagged)[0], good, y[good])
    return out, flagged

over_desp, over_mask = despike_aggressive(raw, threshold=2.0, win=5)

# Peak AREA recovered (subtract the true fluorescence to isolate the bands).
good_area = (despiked - true_fluor).sum()
over_area = (over_desp - true_fluor).sum()
true_area = true_peaks.sum()
print("Over-despiking (thr=2.0, win=5):")
print("  points flagged      : %d of %d (%.0f%%)" % (over_mask.sum(), len(x),
                                                     100 * over_mask.mean()))
print("  real band area kept : good despike %.0f%%  vs  over-despike %.0f%%  (of truth)"
      % (100 * good_area / true_area, 100 * over_area / true_area))

# --- Over-baseline-correction: lambda too small ---------------------------
# A too-flexible baseline bends up under the bands; subtraction then eats the
# bands. With asymmetric weights the symptom is shrinkage, not negative dips.
LAM_BAD = 1e2
base_bad = asls_baseline(despiked, lam=LAM_BAD, p=P)
corr_bad = despiked - base_bad
print("Over-baseline (λ=%.0e instead of %.0e):" % (LAM_BAD, LAM))
print("  1000 cm-1 band height : truth %.3f  good %.3f  over-corrected %.3f (-%.0f%%)"
      % (true_peaks[band], corrected[band], corr_bad[band],
         100 * (1 - corr_bad[band] / true_peaks[band])))
print("  RMSE vs TRUE bands    : good %.4f  vs  over-corrected %.4f"
      % (corr_rmse, np.sqrt(np.mean((corr_bad - true_peaks) ** 2))))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

ax = axes[0]
sl = slice(band - 90, band + 90)
ax.plot(x[sl], true_peaks[sl], lw=1.8, color="#d62728", ls="--", label="true band")
ax.plot(x[sl], (despiked - true_fluor)[sl], lw=1.4, color="#2ca02c", label="good despike")
ax.plot(x[sl], (over_desp - true_fluor)[sl], lw=1.4, color="#9467bd",
        label="over-despiked (eroded)")
ax.set_title("Over-despiking erodes real band area")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label); ax.legend()

ax = axes[1]
ax.plot(x, corrected, lw=0.9, color="#2ca02c", label=f"good baseline (λ={LAM:.0e})")
ax.plot(x, corr_bad, lw=0.9, color="#9467bd", label=f"over-corrected (λ={LAM_BAD:.0e})")
ax.axhline(0, color="k", lw=0.6)
ax.set_title("Over-baseline-correction shrinks the bands toward zero")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label); ax.legend()

fig.tight_layout()
fig.savefig(EXPORTS / "08_failure_cases.png", dpi=150)
plt.show()

Over-despiking flagged a third of the spectrum and shaved real band area;
over-baseline-correction pulled the band tops down toward zero, gutting heights
and areas. **Shrunken peaks are the signature of over-cleaning** (with symmetric
baseline fits you also see tell-tale negative dips) — if you see them, back off
the aggressiveness. The honest target is *artifacts gone, chemistry untouched*,
and the only way to know you've hit it on real data is to watch these symptoms,
because real data won't hand you the answer key.

## 13. A small reusable pipeline

Wrap the correct order into one function you can carry into later lessons:
**despike, then estimate-and-subtract the fluorescence baseline**, returning the
cleaned spectrum plus a few diagnostics so you can sanity-check every run.

In [ ]:
def clean_raman(y, spike_threshold=7.0, lam=1e6, p=0.01):
    '''Clean a raw Raman spectrum: despike first, then remove fluorescence.

    Returns the cleaned spectrum and a dict of diagnostics (spike count, the
    estimated baseline, and the most-negative point -- a quick over-correction
    alarm).
    '''
    despiked, spike_mask = despike(y, threshold=spike_threshold)
    base = asls_baseline(despiked, lam=lam, p=p)
    cleaned = despiked - base
    diagnostics = {
        "n_spikes_removed": int(spike_mask.sum()),
        "baseline": base,
        "min_value": float(cleaned.min()),       # large-negative => over-corrected
    }
    return cleaned, diagnostics


cleaned, diag = clean_raman(raw)
end_to_end_rmse = np.sqrt(np.mean((cleaned - true_peaks) ** 2))
print("pipeline diagnostics :", {k: v for k, v in diag.items() if k != "baseline"})
print("cleaned vs TRUE bands : RMSE = %.4f  (band height %.2f, noise sigma 0.02)"
      % (end_to_end_rmse, true_peaks.max()))

One call, the right order, and a result within detector-noise distance of the
true chemistry — with diagnostics that would flag over-correction on data where
you *can't* peek at the truth.

## Key Takeaways

- **Raman is a weak signal between two strong artifacts.** Fluorescence (broad,
  smooth, from laser-excited emission) and cosmic rays (sharp, single-pixel, from
  particles hitting the CCD) are both *physically real* and both routinely
  *larger* than the bands you want.
- **Width is the discriminator.** A cosmic ray is impossibly narrow; fluorescence
  is impossibly broad; a real band sits in between. Every method here is just a
  way of acting on that width gap.
- **Despiking and baseline correction are different problems.** Despiking is a
  local, high-frequency outlier fix; baseline correction is a global,
  low-frequency smooth fit. Each is blind to the other's target — you need both.
- **Detect spikes by the z-score of the difference.** The point-to-point
  difference amplifies narrowness; a robust (median/MAD) z-score amplifies
  intensity. Repair by interpolating across the flagged pixels *only*.
- **Remove fluorescence with AsLS.** A smoothness penalty plus asymmetric weights
  fits a curve that glides under the bands; subtract it.
- **Order matters: despike first.** Any smoothing or fitting done before
  despiking can spread a spike into a fake band or be biased by it.
- **Over-cleaning is worse than under-cleaning** — it deletes real chemistry.
  Shrunken peaks and negative dips are the warning signs.
- **Ground truth makes it gradeable.** Because the simulator records the true
  bands, background, and spike indices, every step was scored, not eyeballed.

## Practical Checklist

1. **Inspect the raw spectrum first.** Identify the broad swell (fluorescence)
   and any sharp spikes before touching anything.
2. **Despike before everything else.** Use the z-score of the first difference
   with a threshold around 6–8; repair only the flagged pixels by interpolation.
3. **Confirm the despiker was a scalpel.** Verify non-flagged points are
   unchanged and that real band heights/areas survive.
4. **Then estimate the baseline** with AsLS: start near `λ≈1e6`, `p≈0.01`, and
   tune so the baseline passes *under* the bands without bending into them.
5. **Subtract and check the symptoms:** the corrected baseline should be flat at
   zero in band-free regions, with no deep negative dips.
6. **Average multiple exposures if you can.** Cosmic rays land at random
   positions, so they rarely repeat — a median over scans removes most for free.
7. **Record the noise level** (a band-free region's standard deviation) so you
   know what residual error is acceptable.
8. **Keep your raw data.** Every cleaning step is irreversible; archive the
   originals.

## Common Mistakes

- **Smoothing before despiking.** Spreads one-pixel spikes into multi-pixel bumps
  the despiker can no longer catch — they survive as fake bands. *Despike first.*
- **Using one tool for both jobs.** A baseline fit ignores spikes; a despiker
  can't see a slow swell. Reaching for either alone leaves half the mess.
- **Median-filtering the whole spectrum to "kill spikes."** A blanket smoother
  rounds off every real band. Target only the flagged pixels.
- **Threshold too low / replacement window too wide.** Flags real peak flanks and
  gouges out genuine signal — over-despiking.
- **`λ` too small in AsLS.** The baseline bends into the peaks; subtraction
  carves bands away, shrinking heights and areas — over-baseline-correction.
- **Trusting the tallest feature.** In raw Raman the tallest feature is often a
  cosmic ray, not the strongest band.
- **Forgetting fluorescence is sample-dependent.** Change excitation wavelength,
  sample, or substrate and the background changes; re-tune, don't reuse blindly.

## Reporting Guidance

When you publish or hand off cleaned Raman data, state **exactly what you did**,
so a reviewer can reproduce it and judge whether you over-cleaned:

- **Despiking:** method (e.g. modified z-score of the first difference),
  threshold, repair method (interpolation), and **how many points** were altered.
- **Baseline correction:** algorithm (AsLS), the parameters used (`λ`, `p`,
  iterations), and the spectral regions treated as band-free.
- **Order of operations** (despike → baseline → any smoothing), since it changes
  the result.
- **Excitation wavelength and acquisition** (laser line, power, exposure, number
  of accumulations) — these set how much fluorescence and how many cosmic rays
  you start with.
- **A before/after figure** and the residual noise level, so readers can see that
  bands were preserved and no negative artifacts were introduced.
- **Never report only cleaned spectra** with no description of the cleaning — an
  unstated baseline subtraction can manufacture or erase peaks.

## Next Lesson

**6.3 — PCA: Scores, Loadings, and Seeing Structure.** A clean Raman spectrum is
the *input* to chemometrics — and PCA is the gateway. With fluorescence and
cosmic rays removed, the variance that remains is (mostly) chemistry, so PCA
loadings map back to real bands instead of artifacts. We'll take a set of cleaned
spectra, decompose them from scratch with the SVD, and read the loadings *as
molecular information* — the foundation for classification, PLS, and ultimately
spectral **library matching**, where a correctly baseline-corrected, despiked
spectrum is the difference between a confident identification and a false match.